In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib import pyplot as plt
from tqdm import tqdm
import joblib
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
FEATURE_DIR = RESULTS / "features"

In [13]:
country_pseudo_values = pd.read_csv('../data/country-pseudo-values.csv')
region_mask_          = rio.open('../results/region-masks/region-mask-1km.tif')
country_raster_       = rio.open('../results/country-rasters/country-raster-eck4-1km.tif')

In [ ]:
region_mask = rio.open(RESULTS / 'region-masks' / 'region-mask-1km.tif').read(1)

In [ ]:
country_raster        = country_raster_.read(1)

In [23]:
feature_list_1km = dict({
    'elevation': FEATURE_DIR / 'elevation-1km.tif',
    'slope':     FEATURE_DIR / 'slope-1km.tif', 
    'population': RESULTS / 'population-rasters' / 'population-2020-1km.tif',
    'accessibility': FEATURE_DIR / 'accessibility-1km.tif',
    'D2water': FEATURE_DIR / 'distance-to-water-1km.tif', 
    'forest': FEATURE_DIR / 'forest-2020-1km.tif',
    'cropland': FEATURE_DIR / 'cropland-2019-1km.tif', 
    'water': FEATURE_DIR / 'water-mask-1km.tif', 
    'su': FEATURE_DIR / 'su-1km.tif',
    'vb': FEATURE_DIR / 'vb-1km.tif',
    'ss': FEATURE_DIR / 'ss-1km.tif',
    'pb': FEATURE_DIR / 'pb-1km.tif',
    'sm': FEATURE_DIR / 'sm-1km.tif', 
    'sc': FEATURE_DIR / 'sc-1km.tif', 
    'va': FEATURE_DIR / 'va-1km.tif',
    'mt': FEATURE_DIR / 'mt-1km.tif',
    'pa': FEATURE_DIR / 'pa-1km.tif',
    'vi': FEATURE_DIR / 'vi-1km.tif',
    'wb': FEATURE_DIR / 'wb-1km.tif',
    'py': FEATURE_DIR / 'py-1km.tif',
    'pi': FEATURE_DIR / 'pi-1km.tif',
    'ev': FEATURE_DIR / 'ev-1km.tif',
    'nd': FEATURE_DIR / 'nd-1km.tif', 
    'ig': FEATURE_DIR / 'ig-1km.tif', 
    'PA_Ia': RESULTS / 'protected-area-rasters' / 'protected-area-Ia-mask.tif',
    'PA_Ib': RESULTS / 'protected-area-rasters' / 'protected-area-Ib-mask.tif',
    'PA_II': RESULTS / 'protected-area-rasters' / 'protected-area-II-mask.tif',
    'PA_III': RESULTS / 'protected-area-rasters' / 'protected-area-III-mask.tif',
    'PA_IV': RESULTS / 'protected-area-rasters' / 'protected-area-IV-mask.tif',
    'PA_V': RESULTS / 'protected-area-rasters' / 'protected-area-V-mask.tif',
    'PA_VI': RESULTS / 'protected-area-rasters' / 'protected-area-VI-mask.tif',
    'urban': FEATURE_DIR / 'urban-area-2020-1km.tif',
    'bare': FEATURE_DIR / 'bare-land-2020-1km.tif',
    'building_stock': FEATURE_DIR / 'building-mineral-stock-1km.tif',
})

In [5]:
selected_features = ['elevation', 'slope', 'population', 'accessibility', 'D2water', 'forest', 'cropland', 'water', 'su', 'vb', 'ss', 'pb', 'sm', 'sc', 'va', 'mt', 'pa', 'vi', 'wb', 'py', 'pi', 'ev', 'nd', 'ig', 'PA_Ia', 'PA_Ib', 'PA_II', 'PA_III', 'PA_IV', 'PA_V', 'PA_VI', 'urban', 'bare', 'building_stock']
print(len(selected_features)) 

34


In [21]:
region_code = dict({
    'Middle East': 1, 
    'Latin America and the Caribbean': 2,
    'North Africa': 3, 
    'West Africa': 4,
    'East Africa': 5,
    'Central Africa': 6,
    'Southern Africa': 7,
    'Europe': 8, 
    'South-eastern Asia': 9, 
    'Oceania': 10,
    'Southern Asia': 11, 
    'Eastern Asia': 12, 
    'Central and Northern Asia': 13, 
    'Northern America': 14,
    'China': 15
})

In [22]:
pred_global = np.zeros_like(region_mask, dtype=np.float32)

In [ ]:
for i in tqdm(range(1, 256)):
    # print(i)
    if i in [161, 213]:
        continue
    else:
        the_region = country_pseudo_values[country_pseudo_values['value'] == i]['region'].values[0]
        region_id  = region_code[the_region]

    
    model = joblib.load(f'../results/trained-models/xgb-region-{region_id}.pkl')

    country_index = np.where(country_raster == i)

    if len(country_index[0]) == 0:
        # print(f"[Skip] Country {i} has no pixels in raster.")
        continue

    i1, i2 = np.min(country_index[0]), np.max(country_index[0])
    j1, j2 = np.min(country_index[1]), np.max(country_index[1])
    a_window = ((i1, i2), (j1, j2))

    feature_arrays = []
    for feature in selected_features:
        path = feature_list_1km[feature]
        with rio.open(path) as src:
            data = src.read(1, window=a_window).astype(np.float32)
            feature_arrays.append(data.flatten())
    
    X_country = np.stack(feature_arrays, axis=1)
    mask_valid = ~np.isnan(X_country).any(axis=1)
    X_valid = X_country[mask_valid]

    if X_valid.shape[0] == 0:
        continue

    # 预测概率
    p_valid = model.predict_proba(X_valid)[:, 1]

    # 获取窗口的二维形状（从刚读的任意特征 data 得到）
    window_shape = feature_arrays[0].reshape(i2 - i1, j2 - j1).shape

    # 构造完整预测数组并恢复形状
    pred = np.full(X_country.shape[0], np.nan, dtype=np.float32)
    pred[mask_valid] = p_valid
    pred = pred.reshape(window_shape)  # 恢复为窗口形状 (rows, cols)

    # 回填到全局栅格
    submask = (country_raster[i1:i2, j1:j2] == i)
    pred_global[i1:i2, j1:j2][submask] = pred[submask]


In [ ]:
pred_scaled = np.nan_to_num(pred_global, nan=0.0)   # 把NaN替换为0
pred_scaled = np.clip(pred_scaled, 0, 1)            # 限定在0~1之间
pred_scaled = (pred_scaled * 10000).astype(np.uint16)

In [ ]:
with rio.open(
    r'../results/mining-suitability/mining-suitability-1km.tif', 
    'w', 
    driver='GTiff', 
    height=pred_scaled.shape[0],
    width=pred_scaled.shape[1],
    count=1, 
    dtype=pred_scaled.dtype, 
    crs=country_raster_.crs, 
    transform=country_raster_.transform, 
    compress='lzw'
    ) as dst:
    dst.write(pred_scaled, 1)

20260506

In [3]:
# Load the saved global model and its feature names
final_model = joblib.load(RESULTS / 'trained-models' / 'rf-global.pkl')
feature_names_full = joblib.load(RESULTS / 'trained-models' / 'rf-global-features.pkl')

In [14]:
country_pseudo_values = pd.read_csv(DATA / 'country-pseudo-values.csv')

In [18]:
country_raster_       = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [19]:
country_raster        = country_raster_.read(1)

In [12]:
region_mask = rio.open(RESULTS / 'region-masks' / 'region-mask-1km.tif').read(1)

In [24]:
feature_list_1km = dict({
    'elevation': FEATURE_DIR / 'elevation-1km.tif',
    'slope':     FEATURE_DIR / 'slope-1km.tif', 
    'population': RESULTS / 'population-rasters' / 'population-2020-1km.tif',
    'accessibility': FEATURE_DIR / 'accessibility-1km.tif',
    'D2water': FEATURE_DIR / 'distance-to-water-1km.tif', 
    'forest': FEATURE_DIR / 'forest-2020-1km.tif',
    'cropland': FEATURE_DIR / 'cropland-2019-1km.tif', 
    'water': FEATURE_DIR / 'water-mask-1km.tif', 
    'su': FEATURE_DIR / 'su-1km.tif',
    'vb': FEATURE_DIR / 'vb-1km.tif',
    'ss': FEATURE_DIR / 'ss-1km.tif',
    'pb': FEATURE_DIR / 'pb-1km.tif',
    'sm': FEATURE_DIR / 'sm-1km.tif', 
    'sc': FEATURE_DIR / 'sc-1km.tif', 
    'va': FEATURE_DIR / 'va-1km.tif',
    'mt': FEATURE_DIR / 'mt-1km.tif',
    'pa': FEATURE_DIR / 'pa-1km.tif',
    'vi': FEATURE_DIR / 'vi-1km.tif',
    'wb': FEATURE_DIR / 'wb-1km.tif',
    'py': FEATURE_DIR / 'py-1km.tif',
    'pi': FEATURE_DIR / 'pi-1km.tif',
    'ev': FEATURE_DIR / 'ev-1km.tif',
    'nd': FEATURE_DIR / 'nd-1km.tif', 
    'ig': FEATURE_DIR / 'ig-1km.tif', 
    'PA_Ia': RESULTS / 'protected-area-rasters' / 'protected-area-Ia-mask.tif',
    'PA_Ib': RESULTS / 'protected-area-rasters' / 'protected-area-Ib-mask.tif',
    'PA_II': RESULTS / 'protected-area-rasters' / 'protected-area-II-mask.tif',
    'PA_III': RESULTS / 'protected-area-rasters' / 'protected-area-III-mask.tif',
    'PA_IV': RESULTS / 'protected-area-rasters' / 'protected-area-IV-mask.tif',
    'PA_V': RESULTS / 'protected-area-rasters' / 'protected-area-V-mask.tif',
    'PA_VI': RESULTS / 'protected-area-rasters' / 'protected-area-VI-mask.tif',
    'urban': FEATURE_DIR / 'urban-area-2020-1km.tif',
    'bare': FEATURE_DIR / 'bare-land-2020-1km.tif',
    'building_stock': FEATURE_DIR / 'building-mineral-stock-1km.tif',
})

In [7]:
selected_features = ['elevation', 'slope', 'population', 'accessibility', 'D2water', 'forest', 'cropland', 'water', 'su', 'vb', 'ss', 'pb', 'sm', 'sc', 'va', 'mt', 'pa', 'vi', 'wb', 'py', 'pi', 'ev', 'nd', 'ig', 'PA_Ia', 'PA_Ib', 'PA_II', 'PA_III', 'PA_IV', 'PA_V', 'PA_VI', 'urban', 'bare', 'building_stock']
print(len(selected_features)) 

34


In [16]:
region_code = dict({
    'Middle East': 1, 
    'Latin America and the Caribbean': 2,
    'North Africa': 3, 
    'West Africa': 4,
    'East Africa': 5,
    'Central Africa': 6,
    'Southern Africa': 7,
    'Europe': 8, 
    'South-eastern Asia': 9, 
    'Oceania': 10,
    'Southern Asia': 11, 
    'Eastern Asia': 12, 
    'Central and Northern Asia': 13, 
    'Northern America': 14,
    'China': 15
})

In [8]:
# Sanity check — should be 49 (34 env + 15 region one-hot)
print(f'Model expects {len(feature_names_full)} features')
print(f'Region one-hot columns: {[f for f in feature_names_full if f.startswith("reg_")]}')

# Identify region one-hot columns and their order
region_oh_cols = [f for f in feature_names_full if f.startswith('reg_')]
region_ids_in_order = [int(c.split('_')[1]) for c in region_oh_cols]
n_env = len(selected_features)
n_regions = len(region_oh_cols)

Model expects 49 features
Region one-hot columns: ['reg_1', 'reg_2', 'reg_3', 'reg_4', 'reg_5', 'reg_6', 'reg_7', 'reg_8', 'reg_9', 'reg_10', 'reg_11', 'reg_12', 'reg_13', 'reg_14', 'reg_15']


In [9]:
def model_predict_proba_safe(model, X, chunk_size=500_000):
    """Predict in chunks to avoid memory blowup on large countries."""
    n = X.shape[0]
    if n <= chunk_size:
        return model.predict_proba(X)[:, 1]
    out = np.empty(n, dtype=np.float32)
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        out[start:end] = model.predict_proba(X[start:end])[:, 1]
    return out

In [25]:
pred_global = np.zeros_like(region_mask, dtype=np.float32)

for i in tqdm(range(1, 256)):
    if i in [161, 213]:
        continue

    # Look up which region this country belongs to
    matches = country_pseudo_values[country_pseudo_values['value'] == i]
    if len(matches) == 0:
        continue
    the_region = matches['region'].values[0]
    region_id = region_code[the_region]

    # Find this country's bounding window in the raster
    country_index = np.where(country_raster == i)
    if len(country_index[0]) == 0:
        continue

    i1, i2 = np.min(country_index[0]), np.max(country_index[0])
    j1, j2 = np.min(country_index[1]), np.max(country_index[1])
    a_window = ((i1, i2), (j1, j2))

    # Read all 34 environmental features for this window
    feature_arrays = []
    for feature in selected_features:
        path = feature_list_1km[feature]
        with rio.open(path) as src:
            data = src.read(1, window=a_window).astype(np.float32)
            feature_arrays.append(data.flatten())

    X_env = np.stack(feature_arrays, axis=1)   # shape: (n_pixels, 34)
    mask_valid = ~np.isnan(X_env).any(axis=1)
    X_env_valid = X_env[mask_valid]

    if X_env_valid.shape[0] == 0:
        continue

    # Build region one-hot block: every pixel in this country gets the same region
    n_valid = X_env_valid.shape[0]
    region_oh_block = np.zeros((n_valid, n_regions), dtype=np.float32)
    region_col_idx = region_ids_in_order.index(region_id)
    region_oh_block[:, region_col_idx] = 1.0

    # Concatenate to match the model's expected feature order: env + region one-hot
    X_full_valid = np.hstack([X_env_valid, region_oh_block])

    # Predict suitability
    p_valid = model_predict_proba_safe(final_model, X_full_valid)

    # Reshape back to the window
    window_shape = (i2 - i1, j2 - j1)
    pred = np.full(X_env.shape[0], np.nan, dtype=np.float32)
    pred[mask_valid] = p_valid
    pred = pred.reshape(window_shape)

    # Write back to the global array, but only inside this country's pixels
    submask = (country_raster[i1:i2, j1:j2] == i)
    pred_global[i1:i2, j1:j2][submask] = pred[submask]

 36%|███▋      | 93/255 [08:05<14:05,  5.22s/it]  


MemoryError: Unable to allocate 10.5 GiB for an array with shape (82602834, 34) and data type float32

In [ ]:
pred_scaled = np.nan_to_num(pred_global, nan=0.0)
pred_scaled = np.clip(pred_scaled, 0, 1)
pred_scaled = (pred_scaled * 10000).astype(np.uint16)

In [ ]:
with rio.open(
    RESULTS / 'global-mining-suitability-1km.tif',
    'w',
    driver='GTiff',
    height=pred_scaled.shape[0],
    width=pred_scaled.shape[1],
    count=1,
    dtype=pred_scaled.dtype,
    crs=country_raster_.crs,
    transform=country_raster_.transform,
    compress='lzw',
) as dst:
    dst.write(pred_scaled, 1)